In [ ]:
from google.colab import drive
drive.flush_and_unmount()  # 卸载之前的挂载
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
# train_with_labels_visual.py
import os
import re
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader, random_split
from torch.nn.utils.rnn import pad_sequence

# ========== 数据集路径 ==========
ROOT_DIRS = [
    "/content/drive/MyDrive/ml1/dataset/기계학습_1차",

]

# ========== 标签顺序 ==========
ALLOWED_LABELS = ['circle', 'diagonal_left', 'diagonal_right', 'horizontal', 'vertical']

SAVE_DIR = "/content/drive/MyDrive/1/attention_visualization"
os.makedirs(SAVE_DIR, exist_ok=True)

_num_re = re.compile(r"-?\d+(?:\.\d+)?")

def parse_line_to_numbers(line: str):
    nums = []
    parts = line.strip().split(",")
    for p in parts:
        if not p: continue
        found = _num_re.findall(p)
        for f in found:
            nums.append(float(f))
    return nums

def compute_global_feature_dim(paths):
    max_feat = 0
    for p in paths:
        try:
            with open(p, "r", encoding="utf-8", errors="ignore") as f:
                for line in f:
                    nums = parse_line_to_numbers(line)
                    max_feat = max(max_feat, len(nums))
        except:
            pass
    return max_feat if max_feat > 0 else 1

def load_motion_file_padded(path, feature_dim):
    rows = []
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            nums = parse_line_to_numbers(line)
            if len(nums) == 0: continue
            if len(nums) < feature_dim:
                nums += [0.0] * (feature_dim - len(nums))
            rows.append(nums[:feature_dim])
    if len(rows) == 0:
        return np.zeros((1, feature_dim), dtype=np.float32)
    return np.array(rows, dtype=np.float32)

class MotionDataset(Dataset):
    def __init__(self, root_dirs, allowed_labels):
        self.samples = []
        for root in root_dirs:
            for label in os.listdir(root):
                if label not in allowed_labels: continue
                folder = os.path.join(root, label)
                for name in os.listdir(folder):
                    if name.endswith(".txt"):
                        self.samples.append((os.path.join(folder, name), label))
        if len(self.samples) == 0:
            raise RuntimeError("Dataset empty")

        file_paths = [p for p,_ in self.samples]
        self.feature_dim = compute_global_feature_dim(file_paths)

        present = sorted({l for _,l in self.samples})
        self.label_to_idx = {l:i for i,l in enumerate([l for l in ALLOWED_LABELS if l in present])}
        self.idx_to_label = {v:k for k,v in self.label_to_idx.items()}

        print(f"Found {len(self.samples)} files, classes={len(self.label_to_idx)}, global feature_dim={self.feature_dim}")

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        data = load_motion_file_padded(path, self.feature_dim)
        return torch.tensor(data, dtype=torch.float32), torch.tensor(self.label_to_idx[label], dtype=torch.long), path

def collate_fn(batch):
    seqs, labels, paths = zip(*batch)
    seqs = pad_sequence(seqs, batch_first=True)
    return seqs, torch.stack(labels), paths

class LSTMAttnClassifier(nn.Module):
    def __init__(self, input_dim, hidden=128, num_layers=2, num_classes=5, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden, num_layers=num_layers, batch_first=True, dropout=dropout)
        self.attn = nn.Linear(hidden, 1)
        self.fc = nn.Sequential(nn.Linear(hidden, hidden), nn.ReLU(), nn.Linear(hidden, num_classes))

    def forward(self, x):
        outs,_ = self.lstm(x)
        scores = self.attn(outs).squeeze(-1)           # (B,T)
        weights = torch.softmax(scores, dim=1).unsqueeze(-1)
        context = torch.sum(weights * outs, dim=1)
        logits = self.fc(context)
        return logits, weights.squeeze(-1)

def plot_attention(attn, save_path):
    plt.figure(figsize=(12,3))
    plt.imshow(attn[np.newaxis,:], cmap='hot', aspect="auto")
    plt.colorbar()
    plt.yticks([])
    plt.title("Attention Heatmap (Time steps)")
    plt.savefig(save_path)
    plt.close()

def train_and_eval(epochs=30, batch=32, lr=1e-3, val_ratio=0.2):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    ds = MotionDataset(ROOT_DIRS, ALLOWED_LABELS)
    n = len(ds); n_val = int(n * val_ratio); n_train = n - n_val
    train_set, val_set = random_split(ds, [n_train, n_val])

    train_loader = DataLoader(train_set, batch_size=batch, shuffle=True, collate_fn=collate_fn)
    val_loader = DataLoader(val_set, batch_size=batch, shuffle=False, collate_fn=collate_fn)

    model = LSTMAttnClassifier(input_dim=ds.feature_dim, num_classes=len(ds.label_to_idx))
    model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()

    best = 0
    for ep in range(1, epochs+1):
        model.train()
        total, correct, loss_sum = 0, 0, 0
        for X, y, _ in train_loader:
            X,y = X.to(device), y.to(device)
            opt.zero_grad()
            logits,_ = model(X)
            loss = loss_fn(logits,y)
            loss.backward(); opt.step()
            loss_sum += loss.item() * len(y)
            correct += (logits.argmax(1) == y).sum().item()
            total += len(y)
        train_acc = correct / total

        # validation
        model.eval()
        val_correct = 0; val_total = 0
        with torch.no_grad():
            for X,y,_ in val_loader:
                X,y = X.to(device), y.to(device)
                logits,_ = model(X)
                val_correct += (logits.argmax(1) == y).sum().item()
                val_total += len(y)
        val_acc = val_correct / max(val_total,1)

        print(f"Epoch {ep:02d} | Train Acc {train_acc:.4f} | Val Acc {val_acc:.4f}")

        if val_acc > best:
            best = val_acc
            torch.save(model.state_dict(), f"{SAVE_DIR}/best_model.pth")

    print(f"Training finished. Best Val Acc = {best:.4f}")
    return ds, model, device, val_loader

def visualize_val_attention(ds, model, device, val_loader):
    print("\n=== 🔥 保存注意力可视化中 ===")
    model.eval()
    for X, y, paths in val_loader:
        with torch.no_grad():
            X = X.to(device)
            logits, attn = model(X)
            attn = attn.cpu().numpy()
        for a, path in zip(attn, paths):
            name = os.path.basename(path).replace(".txt", "_attn.png")
            save_path = os.path.join(SAVE_DIR, name)
            plot_attention(a, save_path)
    print(f"=== Attention 可视化保存完成：{SAVE_DIR} ===")

if __name__ == "__main__":
    ds, model, device, val_loader = train_and_eval()
    visualize_val_attention(ds, model, device, val_loader)


Found 34 files, classes=5, global feature_dim=28
Epoch 01 | Train Acc 0.2143 | Val Acc 0.0000
Epoch 02 | Train Acc 0.3214 | Val Acc 0.1667
Epoch 03 | Train Acc 0.3571 | Val Acc 0.1667
Epoch 04 | Train Acc 0.4286 | Val Acc 0.1667
Epoch 05 | Train Acc 0.4286 | Val Acc 0.1667
Epoch 06 | Train Acc 0.4286 | Val Acc 0.1667
Epoch 07 | Train Acc 0.4286 | Val Acc 0.1667
Epoch 08 | Train Acc 0.4286 | Val Acc 0.1667
Epoch 09 | Train Acc 0.4286 | Val Acc 0.1667
Epoch 10 | Train Acc 0.4286 | Val Acc 0.1667
Epoch 11 | Train Acc 0.4286 | Val Acc 0.1667
Epoch 12 | Train Acc 0.3571 | Val Acc 0.1667
Epoch 13 | Train Acc 0.3214 | Val Acc 0.1667
Epoch 14 | Train Acc 0.3214 | Val Acc 0.1667
Epoch 15 | Train Acc 0.3571 | Val Acc 0.1667
Epoch 16 | Train Acc 0.4286 | Val Acc 0.1667
Epoch 17 | Train Acc 0.4286 | Val Acc 0.1667
Epoch 18 | Train Acc 0.4286 | Val Acc 0.1667
Epoch 19 | Train Acc 0.4286 | Val Acc 0.1667
Epoch 20 | Train Acc 0.4286 | Val Acc 0.1667
Epoch 21 | Train Acc 0.4286 | Val Acc 0.1667
Epoch 

In [ ]:
# inference_attention.py
import os
import torch
import numpy as np
import matplotlib.pyplot as plt

# =============================
# 配置路径
# =============================
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_PATH = "/content/drive/MyDrive/3/attention_visualization/best_model.pth"
TEST_DIR   = "/content/drive/MyDrive/ml1/dataset/demo"  # 无标签数据
OUTPUT_DIR = "/content/drive/MyDrive/1/attention_visualization/results_unlabeled"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# =============================
# 数据解析
# =============================
def load_motion_file(path):
    data = []
    with open(path, "r") as f:
        for line in f:
            clean = "".join([c if c.isdigit() or c in ['-','.',',','/',' '] else ' ' for c in line])
            items = [x for x in clean.replace("/", " ").replace(",", " ").split() if x.strip()]
            nums = []
            for x in items:
                try: nums.append(float(x))
                except: pass
            if len(nums)>0: data.append(nums)
    if len(data)==0: return None
    max_len = max(len(r) for r in data)
    for i in range(len(data)):
        if len(data[i])<max_len: data[i]+=[0.0]*(max_len-len(data[i]))
    return torch.tensor(data, dtype=torch.float32)

# =============================
# 加载模型
# =============================
checkpoint = torch.load(MODEL_PATH, map_location=DEVICE)
state_dict = checkpoint
label_to_idx = {"circle":0, "diagonal_left":1, "diagonal_right":2, "horizontal":3, "vertical":4}
idx_to_label = {v:k for k,v in label_to_idx.items()}
idx_to_label = {v:k for k,v in label_to_idx.items()}
num_classes = len(label_to_idx)

# 推断 input_dim
sample_files = [f for f in os.listdir(TEST_DIR) if f.endswith(".txt")]
if len(sample_files)==0: raise ValueError("❌ 无 .txt 文件")
tmp = load_motion_file(os.path.join(TEST_DIR, sample_files[0]))
FEATURE_DIM = tmp.shape[1]

# 定义模型
class LSTMAttnClassifier(torch.nn.Module):
    def __init__(self, input_dim, hidden=128, num_layers=2, num_classes=5):
        super().__init__()
        self.lstm = torch.nn.LSTM(input_dim, hidden, num_layers=num_layers, batch_first=True)
        self.attn = torch.nn.Linear(hidden, 1)
        self.fc = torch.nn.Linear(hidden, num_classes)

    def forward(self, x):
        out,_ = self.lstm(x)
        attn_score = torch.softmax(self.attn(out).squeeze(-1), dim=1)
        context = torch.sum(out * attn_score.unsqueeze(-1), dim=1)
        return self.fc(context), attn_score

model = LSTMAttnClassifier(input_dim=FEATURE_DIM, num_classes=num_classes)
model.load_state_dict(state_dict, strict=False)
model.to(DEVICE)
model.eval()

# =============================
# 推理 + Attention 可视化
# =============================
for fname in sample_files:
    fpath = os.path.join(TEST_DIR, fname)
    motion = load_motion_file(fpath)
    if motion is None:
        print(f"⚠ 跳过 {fname}（无有效数据）")
        continue

    motion = motion.unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        logits, attn = model(motion)
        pred = torch.argmax(logits, dim=1).item()
        label = idx_to_label[pred]

    print(f"{fname}: {label}")

    # 保存 Attention 热力图
    attn_np = attn.squeeze(0).cpu().numpy()
    plt.figure(figsize=(8,2))
    plt.imshow(attn_np[np.newaxis,:], cmap="jet", aspect="auto")
    plt.colorbar()
    plt.yticks([])
    plt.title(f"{fname} → {label}")
    save_path = os.path.join(OUTPUT_DIR, fname.replace(".txt","_attention.png"))
    plt.savefig(save_path)
    plt.close()

print("\n🎉 推理完成，Attention 图已保存到:", OUTPUT_DIR)


14.txt: horizontal
12.txt: horizontal
6.txt: horizontal
19.txt: horizontal
7.txt: horizontal
20.txt: horizontal
16.txt: horizontal
11.txt: horizontal
4.txt: horizontal
10.txt: horizontal
3.txt: horizontal
13.txt: horizontal
2.txt: horizontal
1.txt: horizontal
5.txt: horizontal
8.txt: horizontal
17.txt: horizontal
9.txt: horizontal
18.txt: horizontal
15.txt: horizontal

🎉 推理完成，Attention 图已保存到: /content/drive/MyDrive/1/attention_visualization/results_unlabeled


In [ ]:
import os

print("Files in /content/drive/MyDrive/1:")
print(os.listdir("/content/drive/MyDrive/1"))


Files in /content/drive/MyDrive/1:
['attention_visualization']


In [ ]:
import os
TEST_DIR = "/content/drive/MyDrive/ml1/dataset/Machine Learning - Project Data #2"
print(os.listdir(TEST_DIR))


['.DS_Store', 'diagonal_left', 'circle', 'vertical', 'horizontal', 'diagonal_right']


In [ ]:
os.listdir("/content/drive/MyDrive/ml1/dataset")

['Machine Learning - Project Data #2', '기계학습_1차']